### Librerias

In [1]:
import pandas as pd
import openpyxl
from pathlib import Path
import unicodedata

### Concatenacion y Procesamiento en Series

In [5]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

folder = Path(r"data\XM\Generacion")

years = range(2000, 2025)

daily_series = []


# ============================================================
# FUNCIÓN PARA NORMALIZAR TEXTO
# ============================================================

def normalizar_texto(x):
    """
    Convierte textos a mayúsculas, elimina espacios
    y elimina tildes.
    """
    if pd.isna(x):
        return x

    x = str(x).upper().strip()

    x = unicodedata.normalize("NFD", x)
    x = "".join(
        c for c in x
        if unicodedata.category(c) != "Mn"
    )

    return x


# ============================================================
# PROCESAMIENTO DE LOS ARCHIVOS
# ============================================================

for year in years:

    # --------------------------------------------------------
    # Determinar archivos del año
    # --------------------------------------------------------

    if year in [2016, 2017, 2018]:

        files = [
            folder / f"Generacion_(kWh)_{year}SEM1.xlsx",
            folder / f"Generacion_(kWh)_{year}SEM2.xlsx"
        ]

    else:

        files = [
            folder / f"Generacion_(kWh)_{year}.xlsx"
        ]


    # --------------------------------------------------------
    # Procesar cada archivo
    # --------------------------------------------------------

    for file in files:

        # print(f"Procesando: {file.name}")

        # ----------------------------------------------------
        # Leer columnas necesarias
        #
        # A-C  -> Fecha, Recurso, Tipo Generación
        # G-AD -> 24 horas
        # ----------------------------------------------------

        df = pd.read_excel(
            file,
            header=2,
            # usecols="A:C,G:AD"
        )


        # Columnas A, B, C
        info_cols = df.columns[:3]

        # Columnas G en adelante
        hour_cols = df.columns[6:30]

        df = df[
            list(info_cols) + list(hour_cols)
        ]

        # ----------------------------------------------------
        # Identificar columnas
        # ----------------------------------------------------

        date_col = df.columns[0]
        resource_col = df.columns[1]
        type_col = df.columns[2]

        hour_cols = list(df.columns[3:])


        # ----------------------------------------------------
        # Renombrar columnas
        # ----------------------------------------------------

        df = df.rename(
            columns={
                date_col: "Fecha",
                resource_col: "Recurso",
                type_col: "TipoGeneracion"
            }
        )


        # ----------------------------------------------------
        # Convertir fecha
        # ----------------------------------------------------

        df["Fecha"] = pd.to_datetime(
            df["Fecha"],
            errors="coerce"
        )

        # Eliminar filas sin fecha
        df = df.dropna(
            subset=["Fecha"]
        )


        # ----------------------------------------------------
        # Convertir columnas horarias a numérico
        # ----------------------------------------------------

        for col in hour_cols:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )


        # ----------------------------------------------------
        # Normalizar tipo de generación
        # ----------------------------------------------------

        df["TipoGeneracion"] = (
            df["TipoGeneracion"]
            .apply(normalizar_texto)
        )


        # ----------------------------------------------------
        # Generación diaria de cada planta
        #
        # Se suman las 24 horas
        # ----------------------------------------------------

        df["GeneracionDiaria"] = (
            df[hour_cols]
            .sum(axis=1, min_count=1)
        )


        # ----------------------------------------------------
        # Promedio diario de todas las plantas
        # ----------------------------------------------------

        daily_total = (
            df.groupby("Fecha")["GeneracionDiaria"]
              .mean()
              .rename("Generacion")
        )


        # ----------------------------------------------------
        # Promedio diario de plantas HIDRÁULICAS
        # ----------------------------------------------------

        df_hidraulica = df[
            df["TipoGeneracion"] == "HIDRAULICA"
        ]

        daily_hidraulica = (
            df_hidraulica
            .groupby("Fecha")["GeneracionDiaria"]
            .mean()
            .rename("GeneracionHidraulica")
        )


        # ----------------------------------------------------
        # Promedio diario de plantas TÉRMICAS
        # ----------------------------------------------------

        df_termica = df[
            df["TipoGeneracion"] == "TERMICA"
        ]

        daily_termica = (
            df_termica
            .groupby("Fecha")["GeneracionDiaria"]
            .mean()
            .rename("GeneracionTermica")
        )


        # ----------------------------------------------------
        # Unir las tres series diarias
        # ----------------------------------------------------

        daily = pd.concat(
            [
                daily_total,
                daily_hidraulica,
                daily_termica
            ],
            axis=1
        ).reset_index()


        # ----------------------------------------------------
        # Guardar resultado diario
        # ----------------------------------------------------

        daily_series.append(daily)


# ============================================================
# UNIR TODOS LOS ARCHIVOS
# ============================================================

generation_daily = (
    pd.concat(
        daily_series,
        ignore_index=True
    )
    .sort_values("Fecha")
    .reset_index(drop=True)
)


# ============================================================
# ASEGURAR QUE NO HAYA DUPLICADOS
# ============================================================

generation_daily = (
    generation_daily
    .groupby("Fecha", as_index=False)
    .agg({
        "Generacion": "mean",
        "GeneracionHidraulica": "mean",
        "GeneracionTermica": "mean"
    })
)


# ============================================================
# CONSTRUIR SERIE MENSUAL
# ============================================================

generation_monthly = (
    generation_daily
    .set_index("Fecha")
    .resample("MS")
    .sum()
    .reset_index()
)


# ============================================================
# RENOMBRAR FECHA
# ============================================================

generation_monthly = (
    generation_monthly
    .rename(columns={"Fecha": "date"})
    .sort_values("date")
    .reset_index(drop=True)
)


KeyboardInterrupt: 

### Verificacion:

In [6]:
generation_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  300 non-null    datetime64[us]
 1   Generacion            300 non-null    float64       
 2   GeneracionHidraulica  300 non-null    float64       
 3   GeneracionTermica     300 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 9.5 KB


In [7]:
generation_monthly.describe()

,date,Generacion,GeneracionHidraulica,GeneracionTermica
count,300,3.000000e+02,3.000000e+02,3.000000e+02
mean,2012-06-16 02:14:24,3.943166e+07,3.351191e+07,4.207909e+07
min,2000-01-01 00:00:00,1.915901e+07,0.000000e+00,0.000000e+00
25%,2006-03-24 06:00:00,3.495211e+07,3.138312e+07,2.693906e+07
50%,2012-06-16 00:00:00,3.946171e+07,3.896913e+07,4.413498e+07
75%,2018-09-08 12:00:00,4.145178e+07,4.214664e+07,6.420803e+07
max,2024-12-01 00:00:00,8.793532e+07,1.199863e+08,9.038728e+07
std,NaN,9.179821e+06,2.068127e+07,2.646422e+07


In [8]:
generation_monthly.head()

,date,Generacion,GeneracionHidraulica,GeneracionTermica
0,2000-01-01,7.944395e+07,1.073115e+08,5.949018e+07
1,2000-02-01,7.146242e+07,9.742126e+07,5.317335e+07
2,2000-03-01,8.386839e+07,1.096181e+08,6.560681e+07
3,2000-04-01,7.508375e+07,1.024503e+08,5.356420e+07
4,2000-05-01,8.722126e+07,1.156372e+08,6.131389e+07


### Guardado de las series

In [12]:
output_folder = Path(r"data\\XM\Procesadas")

output_folder.mkdir(parents=True, exist_ok=True)

generation_monthly.to_csv(
    output_folder / "generacionOJO_monthly_2000-2024.csv",
    index=False
)